[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/purva-thakre/qce26-qrisp/blob/Pietro_notebooks_Colab_links/Session_2_Introduction_to_Jasp/BonusMaterial/StableHLOExport.ipynb)

# StableHLO Export

Instead of tracing all the way to machine code, in this notebook we stay at the StableHLO layer, reading it, asking the compiler what it costs, and exporting it as a standalone artifact we can ship elsewhere.

## What We Can Inspect: Reading the StableHLO

Let's define an `f` function as a small linear layer (matmul + bias + tanh) that we'll reuse for the rest of the notebook.

We already saw the jaxpr in `JaspTracing.ipynb`. Here we go one level further down, to the StableHLO that `jax.jit` actually hands to the compiler.

In [ ]:
import jax
import jax.numpy as jnp


def f(x, w, b):
    return jnp.tanh(x @ w + b)

Trace `f` first, to get its jaxpr:

In [ ]:
print(jax.make_jaxpr(f)(jnp.ones((4, 8)), jnp.ones((8, 4)), jnp.ones(4)))

{ lambda ; a:f32[4,8] b:f32[8,4] c:f32[4]. let
    d:f32[4,4] = dot_general[
      dimension_numbers=(([1], [0]), ([], []))
      preferred_element_type=float32
    ] a b
    e:f32[1,4] = broadcast_in_dim[
      broadcast_dimensions=(1,)
      shape=(1, 4)
      sharding=None
    ] c
    f:f32[4,4] = add d e
    g:f32[4,4] = tanh f
  in (g,) }


Same primitives we've seen throughout: `dot_general`, `broadcast_in_dim`, `add`, `tanh`, just now on a function with three arguments instead of one. 

Now the same function one level deeper, as StableHLO:

In [ ]:
lowered = jax.jit(f).lower(jnp.ones((4, 8)), jnp.ones((8, 4)), jnp.ones(4))

print(lowered.as_text())

module @jit_f attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<4x8xf32>, %arg1: tensor<8x4xf32>, %arg2: tensor<4xf32>) -> (tensor<4x4xf32> {jax.result_info = "result"}) {
    %0 = stablehlo.dot_general %arg0, %arg1, contracting_dims = [1] x [0], precision = [DEFAULT, DEFAULT] : (tensor<4x8xf32>, tensor<8x4xf32>) -> tensor<4x4xf32>
    %1 = stablehlo.broadcast_in_dim %arg2, dims = [1] : (tensor<4xf32>) -> tensor<1x4xf32>
    %2 = stablehlo.broadcast_in_dim %1, dims = [0, 1] : (tensor<1x4xf32>) -> tensor<4x4xf32>
    %3 = stablehlo.add %0, %2 : tensor<4x4xf32>
    %4 = stablehlo.tanh %3 : tensor<4x4xf32>
    return %4 : tensor<4x4xf32>
  }
}



Observations:

- `dot_general`, `broadcast_in_dim`, `add`, `tanh` all reappear as `stablehlo.*` ops: same computation as the jaxpr, just expressed in the compiler's typed, portable IR.

- Every value now carries an explicit type, `tensor<4x8xf32>` and friends. These are the same shape/dtype information the jaxpr's `f32[4,8]` already carried, just spelled the compiler's way.

## After the Optimizer: Reading the Compiled Program

The `.compile()` method runs the StableHLO through XLA's optimizer. What prints below is target HLO, already scheduled, not StableHLO anymore.

Watch for the 'fusion' optimization: elementwise ops that don't need separate kernels get merged into one, the same idea `XLABackend.ipynb` covered.

In [ ]:
compiled = lowered.compile()
print(compiled.as_text())

HloModule jit_f, is_scheduled=true, entry_computation_layout={(f32[4,8]{1,0}, f32[8,4]{1,0}, f32[4]{0})->f32[4,4]{1,0}}, allow_spmd_sharding_propagation_to_parameters={true,true,true}, allow_spmd_sharding_propagation_to_output={true}

%fused_computation (param_0.1: f32[4,4], param_1.1: f32[4]) -> f32[4,4] {
  %param_0.1 = f32[4,4]{1,0} parameter(0)
  %param_1.1 = f32[4]{0} parameter(1)
  %add.1 = f32[4,4]{1,0} broadcast(%param_1.1), dimensions={1}, metadata={op_name="jit(f)/add" source_file="/tmp/ipykernel_527460/1459353492.py" source_line=6 source_end_line=6 source_column=20 source_end_column=25}
  %add.0 = f32[4,4]{1,0} add(%param_0.1, %add.1), metadata={op_name="jit(f)/add" source_file="/tmp/ipykernel_527460/1459353492.py" source_line=6 source_end_line=6 source_column=20 source_end_column=25}
  ROOT %tanh.0 = f32[4,4]{1,0} tanh(%add.0), metadata={op_name="jit(f)/tanh" source_file="/tmp/ipykernel_527460/1459353492.py" source_line=6 source_end_line=6 source_column=11 source_end_column

Observations:

- The bias-add and `tanh` got fused into a single `%fused_computation`: the compiler decided they didn't need two separate kernels. The matmul (`dot_general`) stays on its own.

- Each op's `metadata` still cites the exact source file, line, and column of the Python that produced it. Even fused, XLA keeps a breadcrumb back to our original code.

## Asking the Compiler What It Costs

The `cost_analysis()` method reports numbers from XLA's static cost model (estimated FLOPs and bytes moved) not a measured wall-clock time.

For a plain `n x n` matmul, we can sanity-check that estimate against the textbook FLOP count.

In [ ]:
n = 256


@jax.jit
def matmul(A, B):
    return A @ B


compiled = jax.jit(matmul).lower(jnp.ones((n, n)), jnp.ones((n, n))).compile()

In [ ]:
cost = compiled.cost_analysis()
print(f"flops reported: {cost['flops']}")
print(f"2*n^3: {2*n ** 3}")
print(f"bytes accessed: {cost['bytes accessed']}")

flops reported: 33554432.0
2*n^3: 33554432
bytes accessed: 786432.0


Observations:

- The reported flops match `2*n**3` exactly, confirming this is a cost-model estimate derived from the computation graph, not a timing measurement.

- `bytes accessed` (786432) is exactly three 256x256 float32 arrays (the two inputs plus the output, 262144 bytes each) nothing extra moved for a single matmul.

- The Same tool works on any jitted function: next, a chain of elementwise ops, to see what fusion (`XLABackend.ipynb`) does to the bytes moved.

In [ ]:
@jax.jit
def chain(x):
    x = jnp.sin(x)
    x = jnp.exp(x)
    x = x * 2.0
    return jnp.tanh(x)

Same `cost_analysis()`, now on `chain` instead of a single matmul:

In [ ]:
x = jnp.ones((1000, 1000))  # 4 MB of float32

cost = jax.jit(chain).lower(x).compile().cost_analysis()
print(f"bytes accessed: {cost['bytes accessed'] / 1e6:1f} MB")

bytes accessed: 8.000000 MB


- 8 MB is exactly one read + one write of the 4 MB tensor: `sin`, `exp`, `*2`, `tanh` all fuse into a single kernel, so the intermediates never actually hit memory.

In [ ]:
compiled_chain = jax.jit(chain).lower(x).compile()
mem = compiled_chain.memory_analysis()
print(f"arguments: {mem.argument_size_in_bytes}")
print(f"output: {mem.output_size_in_bytes}")
print(f"temporaries: {mem.temp_size_in_bytes}")

# dir(mem)

arguments: 4000000
output: 4000000
temporaries: 0


- `memory_analysis` breaks the same kind of estimate down by role: one 4 MB argument (`x`), one 4 MB output, and `temporaries: 0`, confirming what `cost_analysis` already implied, fusion means no intermediate buffer ever gets materialized.

## Model Export with jax.export

`jax.export` compiles a function once into a self-contained artifact you can call or serialize later, without needing the original Python source again.

Export only needs shapes and dtypes, not real data. That's what `jax.ShapeDtypeStruct` is for.

In [ ]:
from jax.export import export


@jax.jit
def model(x, params):
    return jnp.tanh(x @ params["w"] + params["b"])

In [ ]:
x_shape = jax.ShapeDtypeStruct((4, 8), jnp.float32)
params_shape = {
    "w": jax.ShapeDtypeStruct((8, 4), jnp.float32),
    "b": jax.ShapeDtypeStruct((4,), jnp.float32),
}

With the shapes declared, export `model` for real:

In [ ]:
exported = export(model)(x_shape, params_shape)

In [ ]:
# dir(exported)

In [ ]:
type(exported)

jax._src.export._export.Exported

Notice that `export(...)` didn't run `model`, it returned an `Exported` object bundling the StableHLO with everything needed to call or serialize it later.

## Inspecting and Calling an Exported Module

Here, `in_avals`/`out_avals` show the flattened signature the exported module actually expects. Flattened because `params` is a dict, i.e. a pytree, not a single array.

In [ ]:
print("Input shapes:")
for aval in exported.in_avals:
    print(f" {aval}")

Input shapes:
 float32[4,8]
 float32[4]
 float32[8,4]


- Notice the order: `x`, then a length-4 array, then an 8x4 array. `params` was `{"w": ..., "b": ...}`, but JAX flattens dict pytrees by sorted key, so `b` comes before `w` regardless of the order you wrote them in the literal.

In [ ]:
print("Output shapes:")
for aval in exported.out_avals:
    print(f" {aval}")

Output shapes:
 float32[4,4]


In [ ]:
import jax.random as jrandom

key = jrandom.PRNGKey(0)
x = jrandom.normal(key, (4, 8))
params = {"w": jrandom.normal(key, (8, 4)), "b": jnp.zeros(4)}

Call the exported module directly:

In [ ]:
result = exported.call(x, params)
print(f"result: {result}")

result: [[ 0.99954206 -0.34840393 -0.9995229   0.27433315]
 [ 0.5852951  -0.36324698  0.9373974  -0.99999833]
 [-0.9998186   0.98156774 -0.99960905  0.98729825]
 [ 0.9038114  -0.99818474  0.7446091  -0.8613436 ]]


Compare against calling `model` directly, on the same inputs:

In [ ]:
reference = model(x, params)
print(f"reference result: {reference}")
print()
print(f"Max diff: {jnp.max(jnp.abs(result - reference))}")

reference result: [[ 0.99954206 -0.34840393 -0.9995229   0.27433315]
 [ 0.5852951  -0.36324698  0.9373974  -0.99999833]
 [-0.9998186   0.98156774 -0.99960905  0.98729825]
 [ 0.9038114  -0.99818474  0.7446091  -0.8613436 ]]

Max diff: 0.0


Notice that `exported.call` isn't approximating `model`, it's the same compiled computation, just invoked through the export API.

## Dynamic Shapes in Export

Every export above was locked to one fixed shape. `jax.export.symbolic_shape` lets a dimension stay symbolic, so one exported module works for any batch size.

Here, `batch` behaves like a normal value you can drop into a shape tuple. JAX just tracks it symbolically instead of as a concrete integer.

In [ ]:
(batch,) = jax.export.symbolic_shape("batch")

In [ ]:
batch

batch

Drop that symbolic dimension into the shape spec and export again:

In [ ]:
exported_dynamic = export(model)(
    jax.ShapeDtypeStruct((batch, 8), jnp.float32),
    {
        "w": jax.ShapeDtypeStruct((8, 4), jnp.float32),
        "b": jax.ShapeDtypeStruct((4,), jnp.float32),
    },
)

In [ ]:
print("Input shapes:")
for aval in exported_dynamic.in_avals:
    print(f" {aval}")

Input shapes:
 float32[batch,8]
 float32[4]
 float32[8,4]


In [ ]:
print("Output shapes:")
for aval in exported_dynamic.out_avals:
    print(f" {aval}")

Output shapes:
 float32[batch,4]


The signature now literally reads `float32[batch,8] -> float32[batch,4]`. One compiled artifact that works for a batch of 4, 400, or anything else.

## When StableHLO Matters in Practice

Three practical uses of everything we said so far: confirming a function compiles cleanly before deployment, diffing the StableHLO to catch unexpected ops, and serializing a compiled artifact to hand off to another process.

In [ ]:
@jax.jit
def production_fn(x):
    return jnp.sqrt(jnp.sum(x**2, axis=-1))

In [ ]:
# Case 1: Checking that a function lowers correctly before deployment
lowered = jax.jit(production_fn).lower(jnp.ones((100, 64)))
print("Lowering succeeded: StableHLO is valid")

Lowering succeeded: StableHLO is valid


In [ ]:
# Case 2: Checking for unexpected operations
print(f"StableHLO: {lowered.as_text()}")

StableHLO: module @jit_production_fn attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<100x64xf32>) -> (tensor<100xf32> {jax.result_info = "result"}) {
    %0 = call @production_fn(%arg0) : (tensor<100x64xf32>) -> tensor<100xf32>
    return %0 : tensor<100xf32>
  }
  func.func private @production_fn(%arg0: tensor<100x64xf32>) -> tensor<100xf32> {
    %0 = stablehlo.multiply %arg0, %arg0 : tensor<100x64xf32>
    %cst = stablehlo.constant dense<0.000000e+00> : tensor<f32>
    %1 = stablehlo.reduce(%0 init: %cst) applies stablehlo.add across dimensions = [1] : (tensor<100x64xf32>, tensor<f32>) -> tensor<100xf32>
    %2 = stablehlo.sqrt %1 : tensor<100xf32>
    return %2 : tensor<100xf32>
  }
}



Notice `multiply`, `reduce` (with `add`), `sqrt`: exactly the three ops a sum-of-squares norm should produce, nothing unexpected sneaking in.

In [ ]:
# !pip install flatbuffers  # needed below by exported.serialize()

In [ ]:
# Case 3: Serializing and deserializing an exported model
exported = export(jax.jit(production_fn))(jax.ShapeDtypeStruct((100, 64), jnp.float32))

x = jnp.ones((100, 64))
reference = production_fn(x)
print(f"reference result: {reference}")
print()

serialized = exported.serialize()
print(f"Serialized bytes: {len(serialized)}")

del exported, production_fn

reloaded = jax.export.deserialize(serialized)
result = reloaded.call(x)
print(f"Results match: {jnp.allclose(reference, result)}")

reference result: [8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8.
 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8.
 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8.
 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8.
 8. 8. 8. 8.]

Serialized bytes: 1380
Results match: True


Here, the `del` right before reloading is the point. The deserialized module still calls correctly with zero access to the original `production_fn`. The serialized bytes are genuinely self-contained.